# Занятие 6, демо 2. Раскрытие масок - отдельный выбор

Язык из четырёх пар: `[a, a, b, b, c, c, d, d]`. Пока оба символа пары под
маской, про её значение не известно ничего; как только открыт один - второй
определён точно.

Раскрываем все восемь позиций за $N_{\mathrm{FE}}$ вызовов сети. Вопрос: влияет
ли на результат то, **какие** позиции попадают в один вызов.

In [ ]:
import torch

torch.set_num_threads(1)

"""Маленький синтетический язык и маскированная модель над ним.

Язык устроен так, что зависимости между позициями сильные: последовательность
состоит из четырёх пар, и внутри пары символы одинаковы. Пока оба символа пары
скрыты, про её значение не известно ничего; как только открыт один - второй
определён точно. На этом и видно, что раскрытие масок - отдельный выбор.
"""
import math

import torch
from torch import nn
from torch.nn import functional as F

VOCAB = 8          # символы 0..7
LENGTH = 8         # четыре пары
MASK = VOCAB       # отдельный символ маски


def sample_language(n, generator):
    """n последовательностей вида [a, a, b, b, c, c, d, d]."""
    halves = torch.randint(VOCAB, (n, LENGTH // 2), generator=generator)
    return halves.repeat_interleave(2, dim=1)


def is_valid(sequences):
    """Выполняется ли правило языка: символы внутри каждой пары совпадают.

    Это проверка попадания в носитель распределения, а не качество генерации:
    модель, всегда выдающая [0,0,0,0,0,0,0,0], получит здесь единицу.
    """
    pairs = sequences.reshape(sequences.shape[0], LENGTH // 2, 2)
    return (pairs[:, :, 0] == pairs[:, :, 1]).all(dim=1)


def bayes_loss():
    """Неустранимая часть обучающих потерь.

    Позиция под маской предсказывается точно, если открыт её сосед по паре, и
    никак, если сосед тоже под маской, - тогда это равномерное распределение по
    VOCAB. При доле масок rate ~ U[0,1] доля скрытых позиций со скрытым соседом
    равна E[rate^2] / E[rate] = 2/3.
    """
    return (2 / 3) * math.log(VOCAB)


class MaskedModel(nn.Module):
    """Предсказывает исходные символы по последовательности с масками."""

    def __init__(self, width=256):
        super().__init__()
        self.embed = nn.Embedding(VOCAB + 1, 32)
        self.net = nn.Sequential(
            nn.Linear(LENGTH * 32, width), nn.SiLU(),
            nn.Linear(width, width), nn.SiLU(),
            nn.Linear(width, LENGTH * VOCAB),
        )

    def forward(self, tokens):
        h = self.embed(tokens).flatten(start_dim=1)
        return self.net(h).reshape(-1, LENGTH, VOCAB)


def train_masked(steps=4000, batch=512, lr=2e-3, seed=0):
    """Обучение: случайно маскируем часть позиций и предсказываем исходные."""
    torch.manual_seed(seed)
    model = MaskedModel()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    g = torch.Generator().manual_seed(seed + 1)
    history = []

    for _ in range(steps):
        clean = sample_language(batch, g)
        rate = torch.rand(batch, 1, generator=g)
        masked = torch.rand(batch, LENGTH, generator=g) < rate
        if not masked.any():                 # при batch=512 практически не бывает
            continue
        tokens = clean.masked_fill(masked, MASK)

        loss = F.cross_entropy(model(tokens)[masked], clean[masked])
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        history.append(loss.item())

    return model, history


def step_sizes(steps):
    """Сколько позиций раскрывается на каждом из шагов."""
    left, sizes = LENGTH, []
    for remaining in range(steps, 0, -1):
        take = math.ceil(left / remaining)
        sizes.append(take)
        left -= take
    return sizes


def unmask(model, n, steps, policy, order_gen, token_gen, trace=False):
    """Генерация: за `steps` вызовов сети раскрываем все позиции.

    `policy` - "random" (равномерный выбор позиций без возвращения) или
    "confidence" (top-k по максимуму softmax). Значение позиции **сэмплируется**
    из предсказанного распределения: если брать argmax, то при полностью
    замаскированном старте весь прогон детерминирован, все строки batch
    оказываются копиями одной траектории, и доля валидных может быть только
    нулём или единицей.

    Генератор порядка и генератор значений разные: иначе смена политики меняет и
    выпавшие символы, и сравнивать политики становится не с чем.

    Возвращает (tokens, together, trace), где together - сколько раз шаг раскрыл
    обе позиции одной пары сразу.
    """
    tokens = torch.full((n, LENGTH), MASK, dtype=torch.long)
    together = torch.zeros(n, dtype=torch.long)
    steps_trace = []

    with torch.no_grad():
        for take in step_sizes(steps):
            hidden = tokens == MASK
            probs = model(tokens).softmax(dim=-1)

            if policy == "random":
                score = torch.rand(n, LENGTH, generator=order_gen)
            elif policy == "confidence":
                score = probs.max(dim=-1).values
            else:
                raise ValueError(f"неизвестная политика: {policy}")
            chosen = score.masked_fill(~hidden, -1.0).topk(take, dim=1).indices

            picked = torch.zeros(n, LENGTH, dtype=torch.bool)
            picked.scatter_(1, chosen, True)
            as_pairs = picked.reshape(n, LENGTH // 2, 2)
            together += (as_pairs[:, :, 0] & as_pairs[:, :, 1]).sum(dim=1)

            drawn = torch.multinomial(probs.reshape(-1, VOCAB), 1,
                                      generator=token_gen).reshape(n, LENGTH)
            tokens = torch.where(picked, drawn, tokens)

            if trace:
                steps_trace.append(sorted(chosen[0].tolist()))

    return tokens, together, steps_trace


def save_masked(path, model, history, seed):
    """Сохраняет веса вместе с тем, чем они являются."""
    torch.save({"state_dict": model.state_dict(), "objective": "masked",
                "seed": seed, "steps": len(history),
                "final_loss": sum(history[-200:]) / 200}, path)


def load_masked(path):
    """Загружает веса и проверяет, что это маскированная модель этого языка."""
    blob = torch.load(path, map_location="cpu", weights_only=False)
    if blob["objective"] != "masked":
        raise ValueError(f"{path}: обучено на '{blob['objective']}'")
    model = MaskedModel()
    model.load_state_dict(blob["state_dict"])
    model.eval()
    return model, blob

In [ ]:
model_a, blob_a = load_masked("masked_seed0.pt")
model_b, blob_b = load_masked("masked_seed1.pt")

print("две модели одного языка, отличаются только seed обучения")
print(f"потери: {blob_a['final_loss']:.4f} и {blob_b['final_loss']:.4f}, "
      f"неустранимые {bayes_loss():.4f}")
print()
for row in sample_language(2, torch.Generator().manual_seed(0)).tolist():
    print("  пример из языка:", row)

## Одинаковый бюджет, разные политики

Раскрытие идёт по одной из двух политик: **случайной** (равномерный выбор
позиций) и **по уверенности** (top-k по максимуму softmax). Значение позиции
сэмплируется из предсказанного распределения.

Считаем долю последовательностей, удовлетворяющих правилу пар, - по пять
прогонов на клетку, среднее и разброс. И рядом - сколько пар в среднем раскрылись
целиком за один вызов.

In [ ]:
from statistics import mean, pstdev

def measure(model, steps, policy, runs=5):
    valid, together = [], []
    for run in range(runs):
        tokens, pairs, _ = unmask(model, 2000, steps, policy,
                                  torch.Generator().manual_seed(1000 + run),
                                  torch.Generator().manual_seed(2000 + run))
        valid.append(float(is_valid(tokens).double().mean()))
        together.append(float(pairs.double().mean()))
    return mean(valid), pstdev(valid), mean(together)

def sizes(steps):
    parts = step_sizes(steps)
    return (f"{parts[0]} x {len(parts)}" if len(set(parts)) == 1
            else "+".join(map(str, parts)))

print(f"{'NFE':>4} {'за вызов':>9} | {'случайно':>16} {'пар вместе':>11}"
      f" | {'модель A':>9} {'модель B':>9}")
for steps in (1, 2, 3, 4, 8):
    r = measure(model_a, steps, "random")
    a = measure(model_a, steps, "confidence")
    b = measure(model_b, steps, "confidence")
    print(f"{steps:>4} {sizes(steps):>9} |"
          f" {r[0]:>9.3f}+-{r[1]:.3f} {r[2]:>11.2f} |"
          f" {a[0]:>9.3f} {b[0]:>9.3f}")

## Читаем таблицу

**Случайная политика ведёт себя понятно.** Доля валидных растёт по мере того, как
падает число пар, раскрытых целиком за один вызов: 4.00 пары - ноль валидных,
ноль пар - почти все. Само число пар вместе здесь чистая комбинаторика расписания
и от модели не зависит вовсе: 4 при одном вызове, $12/7\approx1.71$ при двух.
Столкнувшаяся пара ломается с вероятностью около $1-1/V=0.875$: оба символа
выбираются из почти равномерного распределения, и совпадают они редко.

**А вот столбцы двух моделей расходятся.** При $N_{\mathrm{FE}}=2$ модель A даёт
0.99, модель B - 0.01. Это две модели одного языка с одинаковыми потерями,
отличаются они только seed'ом обучения.

In [ ]:
print("какие позиции раскрываются на каждом вызове (политика по уверенности):")
for name, model in (("A", model_a), ("B", model_b)):
    for steps in (2, 3):
        _, pairs, trace = unmask(model, 1, steps, "confidence",
                                 torch.Generator().manual_seed(1000),
                                 torch.Generator().manual_seed(2000), trace=True)
        shown = "  ->  ".join(str(s) for s in trace)
        print(f"  модель {name}, NFE={steps}:  {shown}   пар вместе: {int(pairs[0])}")

print()
print("уверенность на полностью замаскированном старте (истинная - 1/8 = 0.125):")
with torch.no_grad():
    for name, model in (("A", model_a), ("B", model_b)):
        start = torch.full((1, LENGTH), MASK)
        top = model(start).softmax(-1).max(-1).values[0]
        print(f"  модель {name}: {[round(v, 3) for v in top.tolist()]}")

## Что из этого следует

На пустом старте все восемь позиций равноценны: истинная уверенность каждой равна
$1/8$. Обученные модели выдают 0.127-0.144 - разброс, который не значит ничего.
Но top-k ранжирует именно его. У модели A первый вызов случайно взял по одной
позиции из каждой пары, у модели B - обе позиции пары сразу. Дальше всё
предопределено.

Отсюда три вывода, и ни один из них не про то, «какая политика лучше».

**Сколько позиций за вызов - отдельная ручка.** Она задаёт, сколько позиций
обязаны решаться без взаимного контекста: раскрытый в этом вызове символ не
успевает стать контекстом для соседа по вызову.

**Какие именно позиции - вторая ручка.** Политика **может** компенсировать
параллельность, если ей удаётся развести зависимые позиции: столбец A при
$N_{\mathrm{FE}}=2$ - ровно такой случай, четыре позиции за вызов и 0.99 валидных.
Но гарантии в этом нет, и столбец B показывает противоположный исход при том же
обучении.

**Уверенность - плохой ориентир там, где её не из чего взять.** Она работает,
когда контекст уже что-то говорит о позиции, и вырождается в ранжирование шума,
когда не говорит ничего.

Это и мотивирует более структурные способы ограничить область параллельного
раскрытия. Блочная диффузия идёт дальше политики сэмплирования: она задаёт
авторегрессию между блоками и диффузию внутри блока, то есть меняет саму
факторизацию модели, а не только порядок раскрытия.

И последнее, про метрику. Доля валидных - это попадание в носитель, а не качество
генерации: модель, всегда выдающая `[0,0,0,0,0,0,0,0]`, получила бы здесь
единицу. Столбец A при $N_{\mathrm{FE}}=2$ честен потому, что символы
сэмплируются, а не берутся argmax'ом.